# __Data Collection or extraction__
---
---

Need to import key modules to function such as pandas, numpy etc.

`code`
```python

import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt 

#this ensures that figures are plotted nicely in line in the jupyter notebook
%matplotlib inline
```

```python
#Create a datafram from importing CSV data
df = pd.read_csv("Filename.csv")

#Print the top 5 rows of the data to get a sense of what one is looking at
print(df.head())

#print the name of the columns of the dataframe to know what it is that one is working with
print(df.columns)

```

In [ ]:
#emtpy cell to use

# __Data Preprocessing__
---
---

# __Exploratory Data Analysis__
---
---

## _Basic scatter plot for one feature vs target_

To interogate one feature's effect on observations, create a dummy feature variable as follows:

`code`
```python

#First we specify our entire data set, into variables/features and observations/labels
#note that df is the dataframe
#.values attribute converts the series/dataframe into numpy arrays
X = df["Variables", "Features"].values
y = df["Observations"].values

#Here i am assuming the 2nd column is the feature of interest
X_single_variable = X[:, 1]

#here one confirms the shapes, should see something like (xxx, ) (xxx, )
print(y.shape, X_single_variable.shape)


#at this point, the X_single_variable has the same dimensions as y, but, needs to be 2 dimensional to be accepted by scikit learn. Therefore, we need to reshape
X_single_variable.reshape(-1,1)
#sanity check to see if the reshaping is correct. Should see something like (xxx, 1)
print(X_single_variable.shape)
```

Here is the basic set of code to plot a scatter plot

`code`
```python
#create the scatter plot, with x and y values
plt.scatter(X_single_variable, y)
#write the y axis label
plt.ylabel("y label")
#write the x axis label
plt.xlabel("X label")
#display the figure
plt.show()
```

## _Basic Statistical Knowledge_

### Primer on Total Sum of Squares 

$$
SS_{\text{tot}}
$$


The **Total Sum of Squares**, denoted

$$
SS_{\text{tot}} = \sum_{i=1}^n (y_i - \bar{y})^2,
$$

is the fundamental measure of **how much variation** is present in the data *before fitting any model*.

It answers a simple question:

> **“How spread out are the observed $y_i$ values around the best constant prediction?”**

---

#### 1. Why we subtract the mean

Suppose we are not allowed to use any predictors.  
We must choose a single constant value $c$ to predict every observation $y_i$.

We want to minimize the total squared error:

$$
S(c) = \sum_{i=1}^n (y_i - c)^2.
$$

This is a function of one variable $c$.  
To find the value of $c$ that minimizes $S(c)$, we take the derivative with respect to $c$:

Step 1: Expand the squared term

$$
S(c)
= \sum_{i=1}^n (y_i^2 - 2 y_i c + c^2)
= \sum y_i^2 - 2c\sum y_i + nc^2.
$$

Step 2: Take the derivative

Differentiate with respect to $c$:

$$
\frac{dS}{dc}
= -2\sum y_i + 2nc.
$$

Step 3: Set the derivative equal to zero (first-order condition for a minimum)

$$
-2\sum y_i + 2nc = 0.
$$

Divide through by $2$:

$$
-\sum y_i + nc = 0.
$$

Solve for $c$:

$$
nc = \sum y_i
\quad\Longrightarrow\quad
c = \frac{1}{n}\sum_{i=1}^n y_i.
$$

Thus:

$$
c = \bar{y}.
$$

Step 4: Second-derivative check (to ensure it's a minimum)

$$
\frac{d^2 S}{dc^2} = 2n > 0,
$$

so the solution is indeed a **minimum**.


The constant $c$ that minimizes the total squared error is the **sample mean**:

$$
\bar{y} = \frac{1}{n}\sum_{i=1}^n y_i.
$$

Therefore, deviations from the mean,

$$
y_i - \bar{y},
$$

represent the **smallest possible total deviation** achievable without using any model at all.

That is why the Total Sum of Squares is defined as:

$$
SS_{\text{tot}} = \sum_{i=1}^n (y_i - \bar{y})^2.
$$

It measures the **minimum possible total squared error for a model with no predictors** –  
which makes it the natural baseline against which we compare any regression model.

---

#### 2. Interpretation of $SS_{\text{tot}}$

$$
SS_{\text{tot}} = \text{Total variation in } y.
$$

It measures:

- how noisy the data are  
- how much “room for explanation” there is  
- how much information a model could potentially capture  

If the points cluster tightly around the mean → $SS_{\text{tot}}$ is small.  
If they are widely scattered → $SS_{\text{tot}}$ is large.

In intuitive terms:

**$SS_{\text{tot}}$ is the baseline squared error if you predict everything with the mean.**

---

#### 3. Geometric intuition (no matrices)

Think of $\bar{y}$ as the **center of mass** of the observations.

Then $SS_{\text{tot}}$ measures the **moment of inertia** of the data around that center.

A regression model tries to explain *why* the observations are spread out:

- trends  
- slopes  
- relationships with predictors  

This is why regression later divides $SS_{\text{tot}}$ into explained and unexplained parts—but this decomposition only works if we start from the mean.

---

#### 4. Why not subtract a trend? Why subtract the mean?

Because the mean is the **unique** constant value that minimizes squared error.

Regression analysis always starts from:

- “What do I know if I know nothing else?” → the mean  
- “How much better can my model do than the mean?”

If there *is* a trend in the data, the regression model will detect and capture it.  
The mean provides the correct baseline for comparison.

---

#### Summary

- $SS_{\text{tot}}$ measures **total variation** in the data.  
- It uses deviations from $\bar{y}$ because the mean is the **best constant predictor**.  
- It represents the **baseline amount of error** before fitting any model.  
- Regression later divides this into **explained** and **unexplained** variation.  
- The key identity $SS_{\text{tot}} = SS_{\text{reg}} + SS_{\text{res}}$ relies on this definition.


### Primer on the Regression Sum of Squares 

$$
SS_{\text{reg}}
$$

The **Regression Sum of Squares**, denoted

$$
SS_{\text{reg}} = \sum_{i=1}^n (\hat{y}_i - \bar{y})^2,
$$

measures how much of the total variation in $y$ is **explained by the regression model**.

It answers:

> **“How far do the model’s predictions $\hat{y}_i$ move away from the baseline prediction $\bar{y}$?”**

---

#### 1. Why compare $\hat{y}_i$ to the mean $\bar{y}$?

Before fitting any model, the **best possible constant predictor** is the sample mean:

$$
\bar{y} = \frac{1}{n} \sum_{i=1}^n y_i.
$$

The purpose of regression is to **improve** on this simple baseline.

Thus, the quantity  
$$
\hat{y}_i - \bar{y}
$$  
represents the **model’s improvement** over predicting the mean.

If $\hat{y}_i$ is far from $\bar{y}$, it means:

- the model has detected structure or trend  
- the model produces predictions that reflect variation in $y$  

---

#### 2. Definition and interpretation

$$
SS_{\text{reg}} = \sum_{i=1}^n (\hat{y}_i - \bar{y})^2.
$$

This measures the **explained variation**, meaning:

- how much of the total variability in $y$ is captured by the model
- how different the fitted values are from the mean
- how much structure the model has uncovered

If $SS_{\text{reg}}$ is **large**, the model explains a lot.  
If it is **small**, the fitted model is barely better than predicting $\bar{y}$.

---

#### 3. Relationship to the total and residual sums of squares

Recall:

- $SS_{\text{tot}}$ = total variation in $y$
- $SS_{\text{res}}$ = unexplained variation (residuals) (see primer on $SS_{\text{res}}$)
- $SS_{\text{reg}}$ = explained variation

Under **ordinary least squares with an intercept**, the following identity holds:

$$
SS_{\text{tot}} = SS_{\text{reg}} + SS_{\text{res}}.
$$

Thus:

- $SS_{\text{reg}}$ is the portion of $SS_{\text{tot}}$ **captured** by the regression
- $SS_{\text{res}}$ is the portion that remains **unexplained**

This division of total variation is the foundation of the $R^2$ statistic.

---

#### 4. Geometric intuition (no matrices used)

Think of:

- $\bar{y}$ as the **center of mass** of the data  
- $\hat{y}_i$ as the model’s attempt to explain why each point sits where it does  

Then:

- $(\hat{y}_i - \bar{y})^2$ measures how much the fitted model **pulls** the predicted value away from the mean
- summing these values gives the **inertia explained by the model**

Thus:

**$SS_{\text{reg}}$ is the “model-explained portion” of the data’s spread.**

---

#### 5. Extreme cases

 **If the model is perfect**  
$\hat{y}_i = y_i$, so

$$
SS_{\text{reg}} = SS_{\text{tot}}, \qquad SS_{\text{res}} = 0.
$$

 **If the model explains nothing**  
$\hat{y}_i = \bar{y}$ for all $i$, so

$$
SS_{\text{reg}} = 0, \qquad SS_{\text{res}} = SS_{\text{tot}}.
$$

---

#### Summary

- The regression sum of squares is  
  $$
  SS_{\text{reg}} = \sum (\hat{y}_i - \bar{y})^2.
  $$
- It measures the **variation explained** by the regression model.  
- Large $SS_{\text{reg}}$ means the model captures meaningful structure.  
- Under OLS with an intercept,  
  $$
  SS_{\text{tot}} = SS_{\text{reg}} + SS_{\text{res}}.
  $$
- It is the counterpart to the residual sum and forms the basis of the $R^2$ statistic.


### Primer on Residual Sum of Squares 

$$
SS_{\text{res}}
$$


The **Residual Sum of Squares**, denoted

$$
SS_{\text{res}} = \sum_{i=1}^n (y_i - \hat{y}_i)^2,
$$

measures how much variation **remains unexplained** after fitting a regression model.

It answers:

> **“After using the model’s predictions $\hat{y}_i$, how far off are we from the observed values $y_i$?”**

---

#### 1. What are the residuals?

For each observation:

- $y_i$ = observed value  
- $\hat{y}_i$ = predicted value from the regression model  
- $e_i = y_i - \hat{y}_i$ = residual

Residuals measure the part of $y_i$ that the model fails to explain.

Thus:

$$
SS_{\text{res}} = \sum e_i^2 = \sum (y_i - \hat{y}_i)^2.
$$

---

#### 2. Why squared residuals?

We square the residuals because:

1. To penalize large errors more strongly  
2. To avoid positive and negative errors canceling out  
3. To produce a smooth, differentiable objective  
4. Because squaring aligns with the Gaussian likelihood assumption

OLS — Ordinary Least Squares — **chooses the model parameters that minimize**:

$$
\sum (y_i - \hat{y}_i)^2.
$$

So $SS_{\text{res}}$ is the quantity the regression tries to make as small as possible.

---

#### 3. Interpretation

A **small** $SS_{\text{res}}$ means:

- The model fits the data well  
- Predictions are close to the observations  
- Little variation is left unexplained  

A **large** $SS_{\text{res}}$ means:

- Poor fit  
- The model misses important structure  
- The residuals are large  

In other words:

**$SS_{\text{res}}$ captures the remaining “noise” after the model has tried to explain the data.**

---

#### 4. Relationship to the mean and to total variation

Recall that:

- $SS_{\text{tot}}$ measures total variation around the mean  
- $SS_{\text{res}}$ measures leftover variation after fitting the model  

Since OLS chooses parameters to minimize squared residuals:

$$
SS_{\text{res}} \le SS_{\text{tot}}
$$

whenever the model includes an intercept.  
(The model can always mimic the mean if nothing else helps.)

---

#### 5. Geometric intuition (no matrices used)

A regression prediction $\hat{y}_i$ captures the model’s explanation of $y_i$.

The residual $e_i$ is the “vertical gap” between the data point and the regression curve/line.

Plot each pair $(x_i, y_i)$ and the fitted line:

- The vertical distances are the $e_i$  
- Squaring and summing them gives $SS_{\text{res}}$  

Thus:

**$SS_{\text{res}}$ is literally the total vertical squared distance between the data and the fitted model.**

---

#### Summary

- The residual sum of squares is  
  $$
  SS_{\text{res}} = \sum (y_i - \hat{y}_i)^2.
  $$
- It measures the **unexplained variation** after regression.  
- OLS chooses coefficients that *minimize* this quantity.  
- A good model has a small $SS_{\text{res}}$; a poor one has a large value.  
- It forms one side of the core identity  
  $$
  SS_{\text{tot}} = SS_{\text{reg}} + SS_{\text{res}},
  $$
  under OLS with an intercept.



### Primer on the ANOVA Decomposition

$$
SS_{\text{tot}} = SS_{\text{reg}} + SS_{\text{res}}.
$$

The **ANOVA (Analysis of Variance) decomposition** is the foundational identity in ordinary least squares (OLS) regression:

$$
SS_{\text{tot}} = SS_{\text{reg}} + SS_{\text{res}}.
$$

It states that:

> **Total variation in the data = Variation explained by the model + Variation left unexplained.**

This clean additive relationship holds **only** for OLS with an **intercept term**.

---

#### 1. The three components

We recall the definitions:

**Total Sum of Squares**

$$
SS_{\text{tot}} = \sum_{i=1}^n (y_i - \bar{y})^2
$$

Measures the total variation in the data around the best constant predictor $\bar{y}$.

---

**Residual Sum of Squares**

$$
SS_{\text{res}} = \sum_{i=1}^n (y_i - \hat{y}_i)^2
$$

Measures the variation left unexplained by the regression model.

---

**Regression Sum of Squares**

$$
SS_{\text{reg}} = \sum_{i=1}^n (\hat{y}_i - \bar{y})^2
$$

Measures the variation explained by the model’s predictions.

---

#### 2. Core identity: where it comes from

Start from the simple decomposition for each data point:


The **ANOVA (Analysis of Variance) decomposition** is the foundational identity in ordinary least squares (OLS) regression:

$$
SS_{\text{tot}} = SS_{\text{reg}} + SS_{\text{res}}.
$$

It states that:

> **Total variation in the data = Variation explained by the model + Variation left unexplained.**

This clean additive relationship holds **only** for OLS with an **intercept term**.

---

#### 1. The three components

We recall the definitions:

**Total Sum of Squares**

$$
SS_{\text{tot}} = \sum_{i=1}^n (y_i - \bar{y})^2
$$

Measures the total variation in the data around the best constant predictor $\bar{y}$.

---

**Residual Sum of Squares**

$$
SS_{\text{res}} = \sum_{i=1}^n (y_i - \hat{y}_i)^2
$$

Measures the variation left unexplained by the regression model.

---

**Regression Sum of Squares**

$$
SS_{\text{reg}} = \sum_{i=1}^n (\hat{y}_i - \bar{y})^2
$$

Measures the variation explained by the model’s predictions.

---

#### 2. Core identity: where it comes from

Start from the simple decomposition for each data point:

$$
y_i - \bar{y}
=
(y_i - \hat{y}_i) + (\hat{y}_i - \bar{y})
=
e_i + (\hat{y}_i - \bar{y}),
$$

where $e_i = y_i - \hat{y}_i$ is the residual.

Now square both sides:

$$
(y_i - \bar{y})^2
=
e_i^2
+
(\hat{y}_i - \bar{y})^2
+
2 e_i(\hat{y}_i - \bar{y}).
$$

Summing over $i$:

$$
SS_{\text{tot}}
=
SS_{\text{res}}
+
SS_{\text{reg}}
+
2\sum_{i=1}^n e_i(\hat{y}_i - \bar{y}).
$$

So the decomposition becomes **exact** if we can show:

$$
\sum_{i=1}^n e_i(\hat{y}_i - \bar{y}) = 0.
$$

And this comes directly from the OLS normal equations.

---

#### 3. Why the cross-term is zero (OLS property)

Expand it:

$$
\sum e_i (\hat{y}_i - \bar{y})
=
\sum e_i \hat{y}_i
-
\bar{y}\sum e_i.
$$

OLS with an intercept guarantees:

1. **Sum of residuals is zero**  
   $$
   \sum e_i = 0
   $$

2. **Residuals are orthogonal to fitted values**  
   $$
   \sum e_i \hat{y}_i = 0
   $$

Both follow from taking derivatives of the sum of squared residuals.

Substitute them:

$$
\sum e_i(\hat{y}_i - \bar{y}) = 0 - \bar{y}\cdot 0 = 0.
$$

Thus:

$$
SS_{\text{tot}} = SS_{\text{reg}} + SS_{\text{res}}.
$$

---

#### 4. Interpretation

**$SS_{\text{tot}}$ = How much variation exists before modeling**  

**$SS_{\text{res}}$ = How much variation the model fails to explain**  

**$SS_{\text{reg}}$ = How much variation the model successfully explains**  

The decomposition shows exactly what regression does:

- It pulls predictions $\hat{y}_i$ away from the mean $\bar{y}$,
- explaining part of the overall spread,
- while the leftovers become the residuals.

---

#### 5. Conditions for the identity to hold

This decomposition holds **if and only if**:

- The model is fit by **ordinary least squares**
- The model includes an **intercept**
- $SS_{\text{tot}}$ is defined using deviations from $\bar{y}$

If you remove the intercept or use a different modeling method (e.g., Ridge, LASSO, GLMs), the identity **does not hold**, and the classic definition of $R^2$ needs modification.

---

#### Summary

- The decomposition  
  $$
  SS_{\text{tot}} = SS_{\text{reg}} + SS_{\text{res}}
  $$  
  partitions total variation into explained and unexplained parts.
- It relies on OLS properties: residuals sum to zero and are orthogonal to fitted values.
- It provides the foundation for interpreting how well a regression model captures structure in the data.
- This identity is what makes the $R^2$ statistic meaningful and interpretable.


# _Basic Linear Regression_

### Linear Regression — A Basic Primer

Linear regression models the relationship between one or more input variables $x$ (features) and an output variable $y$ by fitting a linear function:
$$
\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_p x_p.
$$
The goal is to choose the coefficients $w = (w_0, w_1, \dots, w_p)$ so that the predictions $\hat{y}$ are as close as possible to the observed values $y$.

---

#### Residuals and the Residual Sum of Squares (RSS)

For each observation $i$, the **residual** is:

$$
\varepsilon_i = y_i - \hat{y}_i.
$$

We want these residuals to be small. The typical objective is to minimize the **Residual Sum of Squares (RSS)**:

$$
RSS(w) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2.
$$

Linear regression chooses $w$ to minimize $RSS$.

---

#### Why Least Squares? (Gaussian Origin)

Least squares arises naturally if we assume:

$$
y_i = w_0 + w_1 x_{i1} + \cdots + w_p x_{ip} + \varepsilon_i,
$$

where the noise terms are i.i.d. **Gaussian**:

$$
\varepsilon_i \sim \mathcal{N}(0, \sigma^2).
$$

Under this assumption, the likelihood of observing your dataset is:

$$
L(w) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}} 
    \exp\!\left(-\frac{(y_i - \hat{y}_i)^2}{2\sigma^2}\right).
$$

Maximizing $L(w)$ is equivalent to minimizing:

$$
\sum_{i=1}^{n}(y_i - \hat{y}_i)^2,
$$

which is exactly **RSS**.  

Therefore, **ordinary least squares is the maximum likelihood estimator (MLE) under Gaussian noise**.

---

#### Example: Linear Regression with One Feature

Suppose the model is:

$$
\hat{y}_i = w_0 + w_1 x_i.
$$

RSS becomes:

$$
RSS(w_0, w_1) = \sum_{i=1}^{n}(y_i - w_0 - w_1 x_i)^2.
$$

The closed-form solutions for the coefficients are:

$$
w_1 = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}
           {\sum_i (x_i - \bar{x})^2},
\qquad
w_0 = \bar{y} - w_1 \bar{x}.
$$

---

#### Matrix Form (Multiple Linear Regression)

Let:

- $X$ be the $n \times (p+1)$ design matrix with a leading column of 1s (for the intercept),
- $w$ be the $(p+1) \times 1$ vector of coefficients,
- $y$ be the $n \times 1$ vector of outputs.

Then predictions are:
$$
\hat{y} = X w.
$$

RSS in matrix form:
$$
RSS(w) = (y - Xw)^\top (y - Xw).
$$

##### Closed-form OLS solution

Taking the derivative and setting to zero gives:

$$
w = (X^\top X)^{-1} X^\top y.
$$

This is the standard **normal equation** for linear regression.


### Basic Code

Here is the basic set of code for linear regression. As this is exploratory, there is no need to use a train-test split, but, can use the entire data set to just get a feel for the data.

`code`
```python
#Import the linear regression module from sklearn
#Note that sklearn LinearRegression performs OLS (ordinary least squares) under the hood.

from sklearn.linear_model import LinearRegression

#Instantiate a class for linear regression
reg = LinearRegression()

#fit the data
reg.fit(X_single_variable, y)

#get the predictions, so that we get a line of best fit for the data
predictions = reg.predict(X_single_variable)

#plot the data with a line of best fit. The color option here is "b" for blue data points and "r" for a red straight line
plt.scatter(X_single_variable, y, color = 'b')
plt.plot(X_single_variable, predictions, color = 'r')
plt.ylabel("Target observation")
plt.xlabel("Feature variable")
plt.show()
```

# __Model Selection__
---
---



## _Scikit Work Flow_

Basic Scikit learn work flow is:

```python
from sklearn.module import Model
model = Model()
model.fit(X,y)
model.predict(X_new)
```

## _Splitting the Data Set_
Here we split the data into test and training parts in order to train the model, and then, test the model on previously unseen data to asses its performance.

We commonly use 20-30% of a data set as the test set. The random_state argument sets a seed for a random number generator that splits the data. Using the same number when repeating this step allows us to reproduce the exact split and our downstream results. It is best practice to ensure our split reflects the proportion of labels in our data. So if churn occurs in 10% of observations, we want 10% of labels in our training and test sets to represent churn. We achieve this by setting stratify equal to y. train_test_split returns four arrays: the training data, the test data, the training labels, and the test labels. We unpack these into X_train, X_test, y_train, and y_test, respectively.

Example code below. Different mark down cells used for different chunks.

`code`
```python
#First we specify our entire data set, into variables/features and observations/labels
#note that df is the dataframe
#.values attribute converts the series/dataframe into numpy arrays
X = df["Variables", "Features"].values
y = df["Observations"].values


#An alternative way to generate your data sets are:
#Use axis = 1 to ensure that a column is dropped, instead of a row
X = df.drop("Observations", axis=1).values
y = df["Observations"].values

#sanity check to see that the dimensions (number of rows) of the arrays match.
print(X.shape, y.shape)

#sanity check to ensure the data types are the same
print(type(X), type(y))


```

```python
#Import the module that will be used to split the data
from sklearn.model_selection import train_test_split
```

```python
# Now use the function to split the data.
# Setting the test_size argument 0.3 (for example) we use 30% of the data set as the test set.
# The random_state argument sets a seed for a random number generator that splits the data. Using the same number when repeating this step allows us to reproduce the exact split and our downstream results.
# The stratify argument is there to ensure that the split is so that the proportion of values in the sample produced will be the same as the proportion of values in the split. It is best practice to ensure the split reflects the proportion of labels in our data. So, pick this when a dataset is imbalanced. For example, if a type occurs in 10% of observations, we want 10% of labels in our training and test sets to represent the type.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,random_state=21,stratify=y)

## _Classification Problems_
---

Classification problems deal with observations that are not continous variables, but are in categories, e.g. yes/no, hot/medium/cold etc.

### k-Nearest Neighbors

Popular for classifications problems. The idea of k-Nearest Neighbors, or KNN, is to predict the label of any data point by looking at the "k" closest labeled data points and getting them to vote on what label the unlabeled observation should have. KNN uses majority voting, which makes predictions based on what label the majority of nearest neighbors have.

Example code below. Different mark down cells used for different chunks.

`code`

```python
#import the module
from sklearn.neighbors import KNeighborsClassifier
```

```python
#Instantiate the classifier (example using 5 nearest neighbours)
knn = KNeigborsClassifier(n_neigbors=5)

#fit the data
knn.fit(X_train, y_train)
```

```python
#obtain the predicted observations
y_pred = knn.predict(X_test)
```

In [ ]:

#Cell available to start coding


## _Value or Continous Problems_
---

### Linear Regression

`code`
```python
#Note that sklearn LinearRegression performs OLS (ordinary least squares) under the hood.
from sklearn.linear_model import LinearRegression

# Instantiate the linear regression class
reg_full = LinearRegression()

# Do the fit. At this this stage of the machine learning workflow, do this on a full set of feastures/variables to estimate the target/observations
reg_full.fit(X_train, y_train)

#Predict the data using the test feastures/variables
y_pred = reg_full.predict(X_test)

```

# __Model Evaluation__
---
---

## _Clasification Problems_
---

Common metrics of performance are:

Accuracy
***
$$
A = \frac{C}{T}
$$
where $A$ is accuracy, $C$ is correct predictions and $T$ is total observations

### k-Nearest Neighbors

`code`
```python
#Get the accuracy of the knn test
print(knn.score(X_test, y_test))
```

```python
# Obtain systematic data to determine over/under fitting

# An empty dictionary to store training accuracy data
train_accuracies = {}
# An empty dictionary to store test accuracy data
test_accuracies = {}

#create an array for the number of neigbors over which to iterate
neighbors = np.arange(1, 26)

for neighbor in neighbors:
    #Set up the knn clasifier, useing the variable for number of neighbours
    knn = KNeigborsClassifier(n_neighbors=neighbor)
    #fit the data with the training data
    knn.fit(X_train, y_train)
    #obtain the accuracy scores
    train_accuracies[neighbor] = hnn.score(X_train, y_train)
    test_accuracies[neighbor] = hnn.score(X_test, y_test)
```

```python
#plot the accuracies data
plt.figure(figsize=(8, 6))
plt.title("KNN: Varying Number of Neighbours")
plt.plot(neighbors, train_accuracies.values(), label="Training Accuracies")
plt.plot(neighbors, test_accuracies.values(), label="Testing Accuracies")
plt.legend()
plt.xlabel("Number of Neigbours")
plt.ylabel("Accuracy")
plt.show() 
```

In [ ]:
# Empty cell left open for coding

# __Hyperparameter tuning__
---
---

# __Model Deployment__
---
---